# Βήμα 10 – Σύστημα Α: συνάρτηση πρόβλεψης για νέα δεδομένα
Το αρχείο `src/predict_unsw.py` μαζεύει όλο το Σύστημα Α σε ένα σημείο: δίνεις ροές δικτύου, παίρνεις
**επίθεση ή όχι, τύπο, σοβαρότητα και risk score**.

Εδώ:
1. ελέγχουμε ότι η συνάρτηση βγάζει ακριβώς τα αποτελέσματα των notebooks 11 και 12·
2. βλέπουμε πώς χρησιμοποιείται (πολλές ροές, μία ροή, αρχείο CSV)·
3. βλέπουμε τι κάνει όταν τα δεδομένα εισόδου έχουν πρόβλημα.

Χρόνος: κάτω από 1 λεπτό.

In [ ]:
import sys
import json
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score

sys.path.append("../src")
from predict_unsw import UnswDetector

DATA, MODELS = "../data/", "../models/"
detector = UnswDetector(MODELS)
print("Στήλες εισόδου:", detector.features)
print("Κατώφλι επίθεσης:", detector.threshold, "| όρια σοβαρότητας:", detector.cuts)

## 10.1 Έλεγχος: ίδια αποτελέσματα με τα notebooks 11 και 12
Δίνουμε στη συνάρτηση ολόκληρο το test set και συγκρίνουμε με τα νούμερα που έχουμε ήδη καταγράψει.

In [ ]:
test = pd.read_csv(DATA + "UNSW_NB15_test-set.csv")
test["attack_cat"] = test["attack_cat"].str.strip()
with open(MODELS + "unsw_meta.json", encoding="utf-8") as f:
    severity_map = json.load(f)["severity_map"]
true_severity = test["attack_cat"].map(severity_map)

pred = detector.predict(test)                                  # προεπιλογή: κανόνας Γ
pred_a = detector.predict(test, severity_rule="from_type")     # κανόνας Α, για σύγκριση

high = true_severity == "High"
checks = [("label acc", accuracy_score(test["label"], pred["is_attack"]), 0.8583),
          ("attack_cat acc", accuracy_score(test["attack_cat"], pred["attack_type"]), 0.7165),
          ("severity acc – κανόνας Γ", accuracy_score(true_severity, pred["severity"]), 0.7466),
          ("High → Low – κανόνας Γ", ((pred["severity"] == "Low") & high).sum() / high.sum(), 0.0715),
          ("severity acc – κανόνας Α", accuracy_score(true_severity, pred_a["severity"]), 0.7436),
          ("High → Low – κανόνας Α", ((pred_a["severity"] == "Low") & high).sum() / high.sum(), 0.3512)]
table = pd.DataFrame(checks, columns=["μέτρο", "συνάρτηση", "αναμενόμενο"]).set_index("μέτρο")
table["ίδιο"] = np.where((table["συνάρτηση"] - table["αναμενόμενο"]).abs() < 0.0005, "ΝΑΙ", "ΟΧΙ")
print(table.round(4).to_string())

## 10.2 Πώς φαίνεται η έξοδος
Δέκα τυχαίες ροές του test: αριστερά η πραγματική ετικέτα, δεξιά η απάντηση του συστήματος.

In [ ]:
sample = test.sample(10, random_state=7).index
view = pd.concat([test.loc[sample, ["attack_cat"]].rename(columns={"attack_cat": "πραγματικός τύπος"}),
                  true_severity.loc[sample].rename("πραγματική σοβαρότητα"),
                  pred.loc[sample]], axis=1)
print(view.round(3).to_string())

## 10.3 Μία μόνο ροή
Η συνάρτηση δέχεται και ένα απλό `dict` με τις 8 τιμές. Οι επιπλέον στήλες αγνοούνται και η σειρά δεν παίζει ρόλο.

In [ ]:
flow = test.loc[sample[0], detector.features].to_dict()
print("Είσοδος:", flow)
print(detector.predict(flow).round(3).T.to_string(header=False))

## 10.4 Το κατώφλι ως ρύθμιση λειτουργίας
Το κατώφλι μένει 0.5. Όποιος χρησιμοποιεί το σύστημα μπορεί όμως να το αλλάξει, π.χ. για λιγότερους ψευδείς συναγερμούς.
Ο πίνακας δείχνει την ανταλλαγή στο test· **δεν είναι επιλογή κατωφλιού** (αυτή έγινε στο validation, notebook 11).

In [ ]:
rows = {}
for t in [0.3, 0.5, 0.7, 0.9]:
    p = detector.predict(test, threshold=t)
    normal, attack = test["label"] == 0, test["label"] == 1
    rows[t] = {"συναγερμοί": int(p["is_attack"].sum()),
               "επιθέσεις που εντοπίζονται": p.loc[attack, "is_attack"].mean(),
               "ψευδείς συναγερμοί (FPR)": p.loc[normal, "is_attack"].mean()}
thr = pd.DataFrame(rows).T
thr.index.name = "κατώφλι"
thr["συναγερμοί"] = thr["συναγερμοί"].astype(int)
print(thr.round(4).to_string())

## 10.5 Τι γίνεται με λάθος δεδομένα
Η συνάρτηση σταματά με καθαρό μήνυμα αντί να δώσει πρόβλεψη που δεν σημαίνει τίποτα.

In [ ]:
good = test.loc[sample[:3], detector.features]
cases = {"λείπει στήλη": good.drop(columns=["smean"]),
         "κενή τιμή": good.assign(dur=[np.nan, 1.0, 2.0]),
         "κείμενο αντί για αριθμό": good.astype(object).assign(dpkts=["πολλά", 2, 3]),
         "αρνητική τιμή": good.assign(tcprtt=[-1.0, 0.0, 0.0]),
         "άγνωστος κανόνας": good}
for name, df in cases.items():
    try:
        detector.predict(df, severity_rule="λάθος" if name == "άγνωστος κανόνας" else "expected_tuned")
        print(f"{name}: ΔΕΝ εντοπίστηκε πρόβλημα (δεν θα έπρεπε)")
    except (ValueError, TypeError) as e:
        print(f"{name}: {e}")

## 10.6 Από αρχείο CSV
Το ίδιο γίνεται και από το τερματικό, χωρίς notebook. Από τον φάκελο `thesis_ids`:

```
python src/predict_unsw.py data/UNSW_NB15_test-set.csv -o data/unsw_test_predictions.csv
```

Το αρχείο εξόδου περιέχει τις αρχικές στήλες και δίπλα τις προβλέψεις.